<span></span>
<center>
<h1><b>Fall 2026<br/>
CSC-372 Project Part 8<br/></b></h1>

# Self-Attention from Scratch

</center>


<a href="https://colab.research.google.com/github/fahadsultan/csc372/blob/main/Project8.ipynb" target="_blank" align="right">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In this assignment you'll build the core operation of the Transformer, **self-attention**, with *learned* query, key and value projections. Then you'll train it on two small tasks and **look inside it**: for every token, which other tokens does it attend to?

<br/>

The two tasks were chosen so that the correct attention pattern is predictable in advance:

### Task 1: Reverse

Input:  `3 1 4 1 5 9 2 6`

Target: `6 2 9 5 1 4 1 3`

If your implementation is right, you'll see the attention pattern to be **Position-based** i.e. position `i` attends to position `L-1-i`

### Task 2: Max

Input: `3 1 4 1 5 9 2 6`

Target:  `9 9 9 9 9 9 9 9`

If your implementation is right, you'll see the attention pattern
to be **Content-based** i.e. every position attends to the largest token

### Task 3 — Running Max

Input `3 1 4 1 5 9 2 6`

Target `3 3 4 4 5 9 9 9`

At each position, output the largest token seen so far, including the current token. The expected attention pattern is content-based with a causal mask. Position i attends to the largest token among positions 0 through i. For example, at the second 1, the running max is 4 because the 5 and 9 are still ahead.

Use masked_fill to set the scores for future positions to -inf before applying softmax.

<br/>



**Rules:** don't use `torch.nn.MultiheadAttention` or `F.scaled_dot_product_attention` in your own implementation. (We use the latter only to *test* yours.) Fill in every cell marked `# YOUR CODE HERE` and every *Your answer* block. Cells run in a few minutes on a laptop CPU.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## **Question 1**. Scaled Dot-Product Attention

Given

queries $Q \in \mathbb{R}^{B \times N \times d_k}$

keys $~~~~K \in \mathbb{R}^{B \times N \times d_k}$ and

values $~V \in \mathbb{R}^{B \times N \times d_v}$

where

$B$ is batch size, the number of sequences processed at once (e.g., 128 in the training loop).

$N$ is equence length, the number of tokens per sequence (8 in this assignment, since SEQ_LEN = 8).

$d_k$ is key/query dimension, the size of each query and key vector, i.e. the output size of W_q and W_k. Queries and keys need the same size because attention scores are their dot products.

$d_v$ is value dimension, the size of each value vector, which is also the size of each token’s output from attention. It can differ from $d_k$, but the assignment sets them equal ($d_v$ = $d_k$ = 32).


$$\text{scores} = \frac{QK^\top}{\sqrt{d_k}}$$
<br/>
$$A = \text{softmax}_{\text{row}}(\text{scores})$$
<br/>
$$\text{output} = AV$$

Row $i$ of $A$ is a probability distribution: **how much token $i$ (the query) attends to each token $j$ (the keys).** Keep this convention for the whole assignment: **rows = who is looking, columns = who is being looked at.**

<br/>

**Masking:**

If `mask` is given, it is a boolean tensor broadcastable to `(B, N, N)` where `True` means *allowed*. Set disallowed scores to $-\infty$ before the softmax so that they get exactly zero weight.

Implement `scaled_dot_product_attention` to return **both** the output and the attention weights $A$.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    '''
    Q: (B, N, d_k)   K: (B, N, d_k)   V: (B, L, d_v)
    mask: optional bool tensor broadcastable to (B, N, N); True = may attend
    Returns: output (B, N, d_v), weights (B, N, N)
    '''
    # YOUR CODE HERE
    #
    # 1. Assume Q and K have shape (B, L, d_k), V has shape (B, L, d_v), and the boolean mask,
    # if provided, uses True for positions a query is allowed to attend to.

    # 2. Compute one score for each query–key pair. Use K.transpose(-2, -1) to
    # swap the last two dimensions of K, giving it shape (B, d_k, L).
    # Multiply Q by the transposed K using @ or torch.matmul.
    # What shape should the result have? Scale the scores by sqrt(d_k),
    # where d_k is the size of the last dimension of Q.

    # 3. Apply the mask before softmax. masked_fill replaces entries wherever its boolean argument is True.
    # Since True in your mask means allowed, which mask should you pass to fill the disallowed scores with -inf?

    # 4. Turn scores into weights. Apply softmax over the key dimension (dim=-1).
    # For each query, the weights over its allowed keys should sum to 1.

    # Combine the values. Multiply the attention weights by V using @ or torch.matmul.
    # Check that the output has shape (B, L, d_v), then return the output and the weights.

    raise NotImplementedError()
    return output, weights

In [ ]:
# ---- Tests for Part 1 (do not edit) ----
B, N, d_k, d_v = 2, 5, 8, 6
Q, K, V = torch.randn(B, N, d_k), torch.randn(B, N, d_k), torch.randn(B, N, d_v)

out, w = scaled_dot_product_attention(Q, K, V)
assert out.shape == (B, N, d_v), f"output shape {out.shape}"
assert w.shape == (B, N, N), f"weights shape {w.shape}"
assert torch.allclose(w.sum(-1), torch.ones(B, N), atol=1e-5), "rows of A must sum to 1"
assert torch.allclose(out, F.scaled_dot_product_attention(Q, K, V), atol=1e-5), "doesn't match PyTorch"

causal = torch.tril(torch.ones(N, N, dtype=torch.bool))
out_m, w_m = scaled_dot_product_attention(Q, K, V, mask=causal)
assert torch.all(w_m[:, ~causal] == 0), "masked positions must get zero weight"
assert torch.allclose(out_m, F.scaled_dot_product_attention(Q, K, V, attn_mask=causal), atol=1e-5)
print("Part 1 tests passed ✓")



## **Question 2.** Self-Attention layer with Learned Projections

In *self*-attention, queries, keys and values all come from the same input $X \in \mathbb{R}^{B \times L \times d_{model}}$ through **learned** matrices:

<br/>

$$Q = XW_Q,\quad K = XW_K,\quad V = XW_V$$

<br/>

Implement `SelfAttention`:

* In `__init__`, create `self.W_q`, `self.W_k`, `self.W_v` as `nn.Linear(d_model, d_k, bias=False)` (use `d_v = d_k`).

* In `forward`, project `x`, call your `scaled_dot_product_attention`, and return `(output, weights)`.

* Store the most recent weights in `self.last_weights` (detached) so we can visualize them later.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()
        self.last_weights = None

    def forward(self, x, mask=None):
        # YOUR CODE HERE
        raise NotImplementedError()
        self.last_weights = weights.detach()
        return out, weights

In [ ]:
# ---- Tests for Part 2 (do not edit) ----
attn = SelfAttention(d_model=16, d_k=8)
x = torch.randn(3, 7, 16)
out, w = attn(x)
assert out.shape == (3, 7, 8) and w.shape == (3, 7, 7)
assert attn.last_weights is not None and not attn.last_weights.requires_grad
n_params = sum(p.numel() for p in attn.parameters())
assert n_params == 3 * 16 * 8, f"expected {3*16*8} parameters, got {n_params}"
out.sum().backward()
assert all(p.grad is not None for p in attn.parameters()), "W_q, W_k, W_v must all receive gradients"

# Permutation equivariance: shuffling the input tokens shuffles the output the same way
perm = torch.randperm(7)
out_p, _ = attn(x[:, perm])
assert torch.allclose(out_p, out[:, perm], atol=1e-5)
print("Part 2 tests passed ✓")

**Question 2.1.** The last test shows that self-attention is *permutation equivariant*: shuffle the input and the output is shuffled the same way. Why is that a problem for language, and for the Reverse task in particular?

In [ ]:
# *Your answer:*
#
#

## **Question 3.** A tiny attention model and its training step

Our model is deliberately minimal, so that **attention is the only way information moves between positions**:

```
tokens ──► token embedding + learned positional embedding ──► SelfAttention ──► Linear ──► logits over vocabulary
```

There's no residual connection and no MLP, so a position's prediction depends only on what it attends to. Data generators for both tasks are provided.

In [ ]:
VOCAB, SEQ_LEN = 10, 8   # tokens are the digits 0..9

def make_batch(task, batch_size=128, seq_len=SEQ_LEN):
    x = torch.randint(0, VOCAB, (batch_size, seq_len))
    if task == "reverse":
        y = x.flip(dims=[1])
    elif task == "max":
        y = x.max(dim=1, keepdim=True).values.expand(-1, seq_len)
    elif task == "running_max":                     # used in Part 6
        y = torch.cummax(x, dim=1).values
    else:
        raise ValueError(task)
    return x.to(device), y.to(device)

x, y = make_batch("reverse", batch_size=1); print("reverse:", x.tolist(), "->", y.tolist())

x, y = make_batch("max", batch_size=1);     print("max:    ", x.tolist(), "->", y.tolist())

Implement `forward` in `TinyAttentionModel`. If `use_pos=False`, skip the positional embedding (you'll need this in Part 6).

In [ ]:
class TinyAttentionModel(nn.Module):
    def __init__(self, vocab=VOCAB, seq_len=SEQ_LEN, d_model=32, d_k=32, use_pos=True):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        self.attn    = SelfAttention(d_model, d_k)
        self.out     = nn.Linear(d_k, vocab)
        self.use_pos = use_pos

    def forward(self, tokens, mask=None):
        '''tokens: (B, N) ints  ->  logits: (B, N, vocab)'''
        # YOUR CODE HERE
        raise NotImplementedError()
        return logits

Now complete one optimization step inside `train`: forward pass, cross-entropy loss over **every** position, backward pass, optimizer step.

*Hint:* `F.cross_entropy` expects logits of shape `(N, C)` and targets of shape `(N,)`. Reshape accordingly.

In [ ]:
def train(task, steps=3000, lr=3e-3, mask=None, log_every=500, **model_kwargs):
    torch.manual_seed(0)
    model = TinyAttentionModel(**model_kwargs).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(1, steps + 1):
        x, y = make_batch(task)
        # YOUR CODE HERE
        raise NotImplementedError()
        if step % log_every == 0:
            print(f"step {step:5d} | loss {loss.item():.4f} | acc {accuracy(model, task, mask):.3f}")
    return model

@torch.no_grad()
def accuracy(model, task, mask=None, n=1000):
    x, y = make_batch(task, batch_size=n)
    return (model(x, mask).argmax(-1) == y).float().mean().item()

---

## **Question 4**. Visualizing attention, Train and Interpret

**Before you run the next cell**, sketch (on paper, or in words below) what you expect the attention heatmap to look like for the Reverse task. Then train and compare.

The given `plot_attention(weights, tokens, ax=None, title=None)` draws an $N \times N$ heatmap of one example's attention matrix:

* **rows = query** (the token doing the attending), **columns = key** (the token being attended to);
* label each tick with the token *and* its position, e.g. `"7 (p3)"`;
* label the axes, add a colorbar, and fix the color scale to $[0, 1]$ so that plots are comparable;
* *(optional, for full credit on readability)* print the weight inside each cell.

Then `show_examples` (provided) uses your function to plot several examples side by side.

In [ ]:
def plot_attention(weights, tokens, ax=None, title=None):
    '''weights: (L, L) tensor for ONE example; tokens: length-L list of ints'''
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4.5))
    L = len(tokens)
    w = weights.cpu().numpy()
    im = ax.imshow(w, cmap="viridis", vmin=0, vmax=1)
    labels = [f"{t} (p{i})" for i, t in enumerate(tokens)]
    ax.set_xticks(range(L)); ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticks(range(L)); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("key: attended TO")
    ax.set_ylabel("query: attending FROM")
    for i in range(L):
        for j in range(L):
            if w[i, j] > 0.05:
                ax.text(j, i, f"{w[i, j]:.2f}", ha="center", va="center", fontsize=6,
                        color="black" if w[i, j] > 0.5 else "white")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if title:
        ax.set_title(title)
    return ax

@torch.no_grad()
def show_examples(model, task, n=3, mask=None):
    x, y = make_batch(task, batch_size=n)
    pred = model(x, mask).argmax(-1)
    W = model.attn.last_weights
    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 4.6))
    for k, ax in enumerate(axes):
        plot_attention(W[k], x[k].tolist(), ax=ax,
                       title=f"in   {x[k].tolist()}\npred {pred[k].tolist()}")
    fig.suptitle(f"Task: {task}", y=1.03)
    plt.tight_layout(); plt.show()

In [ ]:
# Sanity check on an UNTRAINED model: the attention should look diffuse and patternless
show_examples(TinyAttentionModel().to(device), "reverse", n=2)

In [ ]:
model_rev = train("reverse")
print("final accuracy:", accuracy(model_rev, "reverse"))
show_examples(model_rev, "reverse")

**Question 5.1.** Describe the pattern you see. Does it match your prediction? Does the attention pattern change from one example to the next? What does that tell you about whether this model attends by *position* or by *content*?


In [1]:
# *Your answer:*
#

In [ ]:
model_max = train("max")
print("final accuracy:", accuracy(model_max, "max"))
show_examples(model_max, "max")

**Question 5.2.** Describe the pattern for the Max task and contrast it with Reverse. What happens when the maximum value appears **more than once** in a sequence? Find such an example (re-run the cell if needed) and explain why the prediction is still correct.

*Your answer:*

In [ ]:
#
#
#

**Question 5.3.** For the Max task, inspect `model_max.attn.W_q.weight` or compute the queries for a few inputs. Are the queries at different positions similar or different? Why does that make sense for this task?

*Your answer:*

In [ ]:
#
#
#

---
## **Question 5.** Experiments and reflection

### 5a. Remove positional embeddings
Train the Reverse task with `use_pos=False` and visualize. Report the accuracy and explain the result using your answer to Question 2.1. *(What accuracy would a model get by guessing the most frequent token in the input?)*

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

*Your answer (6a):*

### 5b. Causal masking
In a decoder-only language model, token $i$ may only attend to tokens $j \le i$. Build a **causal mask** of shape `(SEQ_LEN, SEQ_LEN)` and train on the `running_max` task (target at position $i$ = max of tokens $0..i$). Visualize the result.

1. What shape does the mask force on the heatmap?
2. Within that shape, where does each row put its weight?
3. Why would `running_max` be impossible to learn **perfectly** without a mask in this model, even though the unmasked model has access to *more* information?

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

*Your answer (6b):*

In [ ]:
#
#
#